# Pressure and geometry

The half dimension of 1.5D: as the pressure changes, the flux surfaces move and the geometry factors with them. `EquilibriumCoupling` re-solves GVEC before each segment with TORAX's pressure (a B-spline fit in $s = \rho^2$), keeping the boundary and $\iota$, and gives TORAX the new averages.

The W7-X state that comes with slowcooker is GVEC's example at $\langle\beta\rangle \approx 4.8\,\%$; the transport run has a lower pressure, so the surfaces relax towards the vacuum shape.

In [ ]:
import logging

logging.disable(logging.INFO)  # TORAX and GVEC log every restart and state file at INFO

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots  # noqa: F401
from gvec.util.params import read_parameters

from slowcooker import base_config, gvec_geometry, profiles, run_coupled
from slowcooker.data import equilibrium_files, gvec_equilibrium
from slowcooker.equilibrium import EquilibriumCoupling

equilibrium = gvec_equilibrium("w7x")
rho = np.linspace(0.02, 1.0, 30)
initial = gvec_geometry(equilibrium, rho=rho, angles=(32, 32))
config = base_config(initial, density=1e20, heating=5e6, chi=0.5, t_final=0.4, dt=0.05)
coupling = EquilibriumCoupling(
    read_parameters(str(equilibrium_files("w7x")[0])),
    equilibrium.state,
    rho=rho,
    B_0=config["geometry"]["B_0"],
)
data = profiles(run_coupled(config, [coupling], [0.0, 0.2, 0.4]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
initial.grad_rho2.plasma.plot.lineout(x="rho", ax=axes[0], label="bundled state")
for start, geometry in coupling.geometries.items():
    geometry.grad_rho2.plasma.plot.lineout(
        x="rho",
        ax=axes[0],
        label=f"re-solved at {start:.1f} s",
        title=r"$\langle|\nabla\rho|^2\rangle$",
    )
    (geometry.volume_derivative / initial.volume_derivative).plasma.plot.lineout(
        x="rho", ax=axes[1], label=f"{start:.1f} s", title="$V'$ / bundled"
    )
axes[0].legend()
axes[1].legend()
data.pressure_total.plasma.plot.profiles(over="t", ax=axes[2], title="pressure [Pa]")
fig.tight_layout()

## How often to re-solve

The same run with the equilibrium re-solved once (at the start), every 0.2 s and every 0.1 s, against a reference re-solved every 0.05 s. The geometry moves, but at this pressure its effect on the profiles is small: one re-solve leaves an error of a few $10^{-4}$ in $T_e$, and re-solving about once per confinement time brings it below $10^{-4}$.

In [ ]:
def final_state(interval):
    start = gvec_equilibrium("w7x")
    coupling = EquilibriumCoupling(
        read_parameters(str(equilibrium_files("w7x")[0])),
        start.state,
        rho=rho,
        B_0=config["geometry"]["B_0"],
    )
    times = np.arange(0.0, 0.4 + 1e-9, interval)
    return profiles(run_coupled(config, [coupling], times)).isel(t=-1)


reference = final_state(0.05)
for interval in (0.4, 0.2, 0.1):
    state = final_state(interval)
    error = float(np.max(np.abs(state.T_e - reference.T_e)) / reference.T_e[0])
    print(
        f"re-solve every {interval:.2f} s: max |T_e - T_e,ref| / T_e,ref(0) = {error:.1e}"
    )